In [ ]:
# Kaggle: Internet On, thêm dataset QuantStream và chọn GPU.
import subprocess,sys
# Chỉ cài dependencies cần cho training; không cài dashboard trong notebook.
subprocess.check_call([sys.executable,'-m','pip','install','-q','pyarrow>=17,<27','scikit-learn>=1.5,<2'])

In [ ]:
# Nạp snapshot development, không gồm final holdout.
from pathlib import Path
import sys, zipfile, shutil, json

WORK = Path('/kaggle/working/quantstream') if Path('/kaggle/working').exists() else Path.cwd()/'notebook_work'
WORK.mkdir(parents=True,exist_ok=True)
INPUT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd().parent
archives = list(INPUT.rglob('QuantStream_Kaggle_Input.zip'))
if archives:
    with zipfile.ZipFile(archives[0]) as z:
        for name in z.namelist():
            p=Path(name)
            if p.is_absolute() or '..' in p.parts:
                raise ValueError('Invalid archive path')
        z.extractall(WORK)
else:
    # Kaggle may automatically extract a ZIP uploaded as a Dataset.
    manifests=[p for p in INPUT.rglob('manifest.json') if (p.parent/'development.parquet').exists()]
    if len(manifests)!=1:
        raise FileNotFoundError('Attach the Dataset containing QuantStream_Kaggle_Input.zip (or its extracted contents).')
    source=manifests[0].parent
    shutil.copytree(source,WORK/'snapshot',dirs_exist_ok=True)
    shutil.copytree(source.parent/'code',WORK/'code',dirs_exist_ok=True)
SNAPSHOT=WORK/'snapshot'
sys.path.insert(0,str(WORK/'code'))
manifest=json.loads((SNAPSHOT/'manifest.json').read_text())
assert manifest['contains_final_holdout'] is False
print('Snapshot:',SNAPSHOT)
print('Final holdout included:',manifest['contains_final_holdout'])

In [ ]:
# Kiểm tra CUDA trước khi train.
import torch
print('PyTorch:',torch.__version__)
print('CUDA available:',torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
# Giữ nguyên 6 epoch của thí nghiệm.
from quantstream.deep import train_snapshot
OUTPUT=WORK/'tcn_output'
EPOCHS = 6  # Same maximum epoch count as the CPU reference run.
metadata=train_snapshot(SNAPSHOT,OUTPUT,epochs=EPOCHS,device='auto')
print(json.dumps(metadata['selection'],indent=2))

In [ ]:
# Lưu output và metadata.
archive=shutil.make_archive(str(WORK.parent/'TCN_Kaggle_Output'),'zip',OUTPUT)
print('Download:',archive)
from IPython.display import FileLink,display
display(FileLink(archive))

In [ ]:
# Loss theo fold; đường final train không có validation mới.
import pandas as pd
import matplotlib.pyplot as plt
loss_rows = [{'fold': f['fold'] + 1, **e} for f in metadata['history'] for e in f['epochs']]
loss = pd.DataFrame(loss_rows)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for fold, group in loss.groupby('fold'):
    axes[0].plot(group.epoch, group.train_huber, label=f'Fold {fold}')
    axes[1].plot(group.epoch, group.validation_mse, label=f'Fold {fold}')
for ax, title in zip(axes, ['Train Huber (scaled target)', 'Validation MSE (return)']):
    ax.set(title=title, xlabel='Epoch', ylabel='Loss'); ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()